In [ ]:
import logging
import pandas as pd
import numpy as np
import os

logging.basicConfig(level=logging.INFO)

class FileValidationError(Exception):
    pass

class FileMissingValueError(FileValidationError):
    pass

def load_results(filepath: str) -> pd.DataFrame:
    if not os.path.exists(filepath):
        raise FileValidationError("File not found!!")  
    elif os.path.getsize(filepath) == 0:
        raise FileMissingValueError("File exists but contains no value or data")
    else:
        df = pd.read_csv(filepath, encoding='utf-8-sig')
        logging.info(f"Loaded {filepath} — shape: {df.shape}")
        return df

try:
    al = load_results("data/raw/Allocated Limit for Honble MPs.csv")
    wc = load_results("data/raw/Works Completed.csv")
    ws = load_results("data/raw/Works Sanctioned.csv")

except FileValidationError as e:
    logging.error(f"Loading Failed: {e}")
    raise

In [ ]:
import re

al = al.iloc[:-1]
ws = ws.iloc[:-1]
wc = wc.iloc[:-1]

def clean_dataframe_columns(df: pd.DataFrame) -> pd.DataFrame:
    
    dfCols = list(df.columns)
    columns_cleaned = []

    for col in dfCols:
        cleaned_names = re.sub("[\.\'\()\₹]", '',str(col)).lower().strip()
        cleaned_names = cleaned_names.replace(" ", "_")
        columns_cleaned.append(cleaned_names)

    df.columns = (columns_cleaned)
    return df


clean_dataframe_columns(al).columns
clean_dataframe_columns(wc).columns
clean_dataframe_columns(ws).columns

In [ ]:
al.columns, ws.columns, wc.columns

In [ ]:
def diagnose_dataframe(df: pd.DataFrame, name: str):
    logging.info(f"\n Dataframe -> {name}\n")
    logging.info(f" Shape: \n{df.shape}\n")
    logging.info(f" Dataframe Datatypes \n{df.dtypes}\n")
    logging.info(f" Dataframe null columns \n{df.isna().sum()[df.isna().sum()>0]}\n")
    logging.info(f" Datarframe duplicates \n{df.duplicated().sum()}")

    
diagnose_dataframe(al, "Allocated Limit for Honble MPs")
diagnose_dataframe(wc, "Works Completed")
diagnose_dataframe(ws, "Works Sanctioned")

In [ ]:
print(al.isna().sum()[al.isna().sum()>0])
print(wc.isna().sum()[wc.isna().sum()>0])
print(ws.isna().sum()[ws.isna().sum()>0])

In [ ]:
def numeric_conversion(df:pd.DataFrame, columns: list) -> pd.DataFrame:
    for cols in columns:
        df[cols] = pd.to_numeric(df[cols], errors="coerce", downcast="float")
    return df

al = numeric_conversion(al, ["allocated_amount"])
wc = numeric_conversion(wc, ["amount_disbursed"])
ws = numeric_conversion(ws, ["sanction_amount"])

In [ ]:
print(al.isna().sum()[al.isna().sum()>0])
print(wc.isna().sum()[wc.isna().sum()>0])
print(ws.isna().sum()[ws.isna().sum()>0])

In [ ]:
def replace_strings_with_real_nulls(df: pd.DataFrame) -> pd.DataFrame:
    df = df.replace("NaN", np.nan)
    return df
    
al = replace_strings_with_real_nulls(al)
wc = replace_strings_with_real_nulls(wc)
ws = replace_strings_with_real_nulls(ws)

In [ ]:
def strip_whitespace(df: pd.DataFrame) -> pd.DataFrame:
    for col_name in df.columns:
        if df[col_name].dtype == "object":
            df[col_name] = df[col_name].str.strip()
            df[col_name] = df[col_name].str.replace(r"\t", " ", regex=True) 

    return df

al = strip_whitespace(al)
wc = strip_whitespace(wc)
ws = strip_whitespace(ws)

# al.style.format({"allocated_amount": "{:,.2f}"})
# wc.style.format({"amount_disbursed": "{:,.2f}"})
# wc.style.format({"sanction_amount": "{:,.2f}"})
# DO-NOT UNCOMMENT THESE!!

In [ ]:
def convert_real_datetime(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    for cols in columns:
        initial_count =  df[cols].isna().sum()
        df[cols] = pd.to_datetime(df[cols], errors="coerce",  format="%d-%b-%Y")
        failed_count = df[cols].isna().sum()
        failed = failed_count - initial_count
        logging.info(f"Column '{cols}': {failed} unparseable entries turned into NaT.")
    return df

wc_datetime_list = ["completion_date"]
ws_datetime_list = ["recommended_date", "sanction_date"]
wc = convert_real_datetime(wc, wc_datetime_list)
ws = convert_real_datetime(ws, ws_datetime_list)

In [ ]:
def drop_duplicate_rows(df: pd.DataFrame, name: str) -> pd.DataFrame:
    before = df.shape[0]
    df = df.drop_duplicates()
    after = df.shape[0]
    logging.info(f"{name}: dropped {before - after} duplicate rows")
    return df

al = drop_duplicate_rows(al, "Allocated Limit")
ws = drop_duplicate_rows(ws, "Works Sanctioned")
wc = drop_duplicate_rows(wc, "Works Completed")

In [ ]:
al = al.drop(columns=["sr_no"])
ws = ws.drop(columns=["sr_no"])
wc = wc.drop(columns=["sr_no", "image"])

In [ ]:
diagnose_dataframe(al, "Allocated Limit (cleaned)")
diagnose_dataframe(ws, "Works Sanctioned (cleaned)")
diagnose_dataframe(wc, "Works Completed (cleaned)")

In [ ]:
wc

al

In [ ]:
def mp_name_clean(raw_name: str) -> str:
    cleaned_name = re.sub(r"^(Shri|Smt|Dr\.|Er\.)\s*", "", raw_name).upper()
    return cleaned_name

# al2 = al.copy()
# al2["honble_members_of_parliaments"] = al2["honble_members_of_parliaments"].apply(normalize_mp_name)

al["honble_members_of_parliaments"] = al["honble_members_of_parliaments"].apply(mp_name_clean)
ws["honble_members_of_parliament"] = ws["honble_members_of_parliament"].apply(mp_name_clean)
wc["honble_members_of_parliament"] = wc["honble_members_of_parliament"].apply(mp_name_clean)

In [ ]:
al_set = set(al["honble_members_of_parliaments"].unique())
ws_set = set(ws["honble_members_of_parliament"].unique())
wc_set = set(wc["honble_members_of_parliament"].unique())

ws_matches = ws_set.intersection(al_set)
wc_matches = wc_set.intersection(al_set)

sanctioned_match_rate = len(ws_matches) / len(ws_set) if ws_set else 0.0
completed_match_rate = len(wc_matches) / len(wc_set) if wc_set else 0.0

logging.info("--- Overlap Summary ---")
logging.info(f"Sanctioned: {len(ws_matches)} names match Allocated out of {len(ws_set)} total.")
logging.info(f"Sanctioned Match Rate: {sanctioned_match_rate:.2%}")

logging.info(f"Completed: {len(wc_matches)} names match Allocated out of {len(wc_set)} total.")
logging.info(f"Completed Match Rate: {completed_match_rate:.2%}")

if sanctioned_match_rate < 0.90:
    logging.warning(f"Low match rate in Sanctioned! Sample of unmatched names: {list(ws_set - al_set)[:15]}")

if completed_match_rate < 0.90:
    logging.warning(f"Low match rate in Completed! Sample of unmatched names: {list(wc_set - al_set)[:15]}")

In [ ]:
def grouping_by_mps(df: pd.DataFrame, grouping_column: str, amount_column: str) -> pd.DataFrame:
    return df.groupby(grouping_column).agg(
    total_sanction_amount=(amount_column, "sum"),
    sanction_work_count=(amount_column, "count")
)

test_ws = grouping_by_mps(ws, "honble_members_of_parliament", "sanction_amount")
test_wc = grouping_by_mps(wc, "honble_members_of_parliament", "amount_disbursed")

test_wc.rename(columns={"total_sanction_amount": "total_amount_disbursed"}, inplace=True)
# Minor Issuse fixed now.
al.rename(columns={"honble_members_of_parliaments": "honble_members_of_parliament"}, inplace=True)

In [ ]:
step_1 = pd.merge(al, test_ws, on="honble_members_of_parliament", how="left")
final_merged = pd.merge(step_1, test_wc, on="honble_members_of_parliament", how="left")

final_merged = final_merged.rename(columns={
    "sanction_work_count_x": "sanctioned_work_count",
    "sanction_work_count_y": "completed_work_count",
    "total_amount_disbursed": "total_disbursed_amount"
})

cols_to_fill = ["total_sanction_amount", "sanctioned_work_count", "total_disbursed_amount", "completed_work_count"]
final_merged[cols_to_fill] = final_merged[cols_to_fill].fillna(0)

In [ ]:
print(final_merged.shape)
zero_activity = final_merged[(final_merged["sanctioned_work_count"] == 0) & (final_merged["completed_work_count"] == 0)]
print(f"MPs with zero sanctioned and zero completed works: {len(zero_activity)}")
orphans = final_merged[(final_merged["sanctioned_work_count"] == 0) & (final_merged["completed_work_count"] > 0)]
print(f"MPs with completed works but zero matching sanctions: {len(orphans)}")

In [ ]:
df = final_merged.copy()
df = df.copy()
df["utilization_rate"] = df["total_disbursed_amount"] / df["allocated_amount"]
df["sanctioned_backlog"] = df["total_sanction_amount"] - df["total_disbursed_amount"]
df["completion_ratio"] = np.where(df["sanctioned_work_count"] == 0, np.nan, df["completed_work_count"] / df["sanctioned_work_count"])
df["has_no_activity"] = (df["sanctioned_work_count"] == 0) & (df["completed_work_count"] == 0)
# an MP who shows completed, disbursed works in the Completed table, but has zero matching sanctioned works in the Sanctioned table.
df["has_orphan_completions"] = (df["sanctioned_work_count"] == 0) & (df["completed_work_count"] > 0)

In [ ]:
state_summary = df.groupby("state").agg(
        avg_utiliazation_rate=("utilization_rate","mean"),
        avg_sanctioned_backlog=("sanctioned_backlog", "mean")
    ).sort_values(by="avg_utiliazation_rate", ascending=False)

In [ ]:
cutoff_value = 1.47e08
filtered_df = df[df["allocated_amount"] > cutoff_value]
top_10_mps = filtered_df.sort_values(by="utilization_rate", ascending=False).head(10)
bottom_10_mps = filtered_df.sort_values(by="utilization_rate", ascending=True).head(10)

In [ ]:
state_summary.head(10)

In [ ]:
state_summary.tail(10)

In [ ]:
top_10_mps[["honble_members_of_parliament","state","allocated_amount","utilization_rate"]]

In [ ]:
bottom_10_mps[["honble_members_of_parliament","state","allocated_amount","utilization_rate"]]

In [ ]:
df["state"].isna().sum()


In [ ]:
df["state"].unique()

In [ ]:
(df["state"] == "").sum()

In [ ]:
state_summary = df[df["state"] != ""].groupby("state").agg(
    avg_utiliazation_rate=("utilization_rate", "mean"),
    avg_sanctioned_backlog=("sanctioned_backlog", "mean")
).sort_values(by="avg_utiliazation_rate", ascending=False)

In [ ]:
df

In [ ]:
ws = ws[ws["work_category"] != ""]
sanction_by_category = ws.groupby("work_category").agg(
    total_sanction_amount = ("sanction_amount", "sum")
)
disbursed_by_category = wc.groupby("work_category").agg(
    total_disbursed_amount = ("amount_disbursed", "sum")
)

In [ ]:
check = pd.merge(sanction_by_category, disbursed_by_category, on="work_category", how="left")

In [ ]:
check

In [ ]:
ws["work_category"].value_counts()
wc["work_category"].value_counts()

In [ ]:
check["gap"] = check["total_sanction_amount"] - check["total_disbursed_amount"]
check = check.sort_values(by="gap", ascending=False)

In [ ]:
check

In [ ]:
df.sort_values(by="utilization_rate", ascending=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def plot_utlization_rate(df: pd.DataFrame) -> None:
    plt.style.use("bmh")

    plt.hist(df["utilization_rate"], bins=15, color="royalblue", edgecolor="black")
    plt.axvline(
        x = np.mean(df["utilization_rate"]),
        color="#FF2800",
        linestyle="dashdot",
        linewidth=2,
        label="Average Utilization Line"
    )
    plt.xlabel("Utilizaion Rate")
    plt.ylabel("Number of MPs")
    plt.title("Distribution of MPLADS Fund Utilization Rate")
    plt.legend()
    plt.show()

plot_utlization_rate(df)

In [ ]:
def plot_state_boxplot(df: pd.DataFrame) -> None:
    sns.boxplot(data=df, x="state", y="utilization_rate", color="royalblue")
    plt.xticks(rotation=45, ha='right')
    plt.xlabel("State", color="#FF2800")
    plt.ylabel("Utilizaion Rate", color="#FF2800")
    plt.tight_layout()
    plt.show()

plot_state_boxplot(df)

In [ ]:
corr_matrix =  df[["allocated_amount", "total_sanction_amount", "total_disbursed_amount", "utilization_rate"]].corr()
sns.heatmap(
    corr_matrix, 
    annot=True
)
plt.title("Correlation Heatmap: MPLADS Fund Metrics")
plt.show()

# Low correlation between allocated_amount and total_disbursed_amount
# → more funds allocated does NOT mean more funds get utilized/completed.
# Confirms zero-activity MP finding: money alone isn't the bottleneck.

In [ ]:
state_total = df.groupby("state")["allocated_amount"].sum().divide(1e7).sort_values(ascending=False).head(10).reset_index()

ax = sns.barplot(
    data=state_total, 
    x="allocated_amount", 
    y="state",
    hue="allocated_amount",
    palette="ch:start=.2,rot=-.3",
)

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=5)

plt.xlabel("Allocated Amount (in crores)")
plt.ylabel("States")
plt.title("States with the highest amount allocation")
plt.tight_layout()
plt.show()

In [ ]:
sns.scatterplot(
    data=df,
    x="allocated_amount", 
    y="total_disbursed_amount", 
    hue="has_no_activity", 
    style="has_no_activity"
)

plt.xlabel("Allocated Amount")
plt.ylabel("Total Disbursed Amount")
plt.title("Amount Allocated vs Total Amount Disbursed")
plt.legend(
    title = "Has no activity",
)
plt.show()

In [ ]:
df["utilization_rate"].isna().sum()
df["state"].isna().sum()

df = df.dropna(subset=["utilization_rate", "state"])

In [ ]:
class MissingColumnError(Exception):
    """Must be triggered if utilization rate column is missing"""
    pass

def defined_completion_risk(df: pd.DataFrame, method: str, threshold: int=None, group_col: str = None):
    df = df.copy()
    if "utilization_rate" not in df.columns:
        raise MissingColumnError("Column utilization rate doesn't exist in the dataframe!")

    if (method == "threshold"):
        df["at_risk"] = (df["utilization_rate"] <= threshold).astype(int)

    elif(method == "relative"):
        group_medians = df.groupby(group_col)["utilization_rate"].transform("median")
        df["at_risk"] = (df["utilization_rate"] <= group_medians).astype(int)
        
    else:
        raise ValueError(f"Unknown method '{method}'. Choose either 'threshold' or 'relative'.")
        
    return df

defined_completion_risk(df, method="relative", group_col="state")["at_risk"].value_counts(normalize=True)*100

In [ ]:
df.columns

In [ ]:
df["completion_ratio"] = df["completion_ratio"].fillna(0)
df["has_no_activity"] = df["has_no_activity"].astype(int)
df["has_orphan_completions"] = df["has_orphan_completions"].astype(int)

LEAKAGE_COLS = ["utilization_rate"]
ID_COLS = ["honble_members_of_parliament", "constituency"]
numeric_features = ["allocated_amount", "total_sanction_amount", "sanctioned_work_count", "total_disbursed_amount", "completed_work_count", "sanctioned_backlog", "completion_ratio", "has_no_activity", "has_orphan_completions"]
categorical_features = ["state"]
df[numeric_features + categorical_features].isnull().sum()

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

def build_feature_pipeline(numeric_features: list, categorical_features: list) -> ColumnTransformer:
    col_trasnform = ColumnTransformer(
        transformers = [
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
        ]
    )
    return col_trasnform

X = df[numeric_features + categorical_features]
pipeline = build_feature_pipeline(numeric_features, categorical_features)
X_transformed = pipeline.fit_transform(X)
X_transformed.shape

In [ ]:
print(X_transformed[np.isnan(X_transformed)].sum())

In [ ]:
import logging
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

from src.risk_target import defined_completion_risk

from src.feature_pipeline import build_feature_pipeline

class ModelNotTrainedError(Exception):
    """raise this if user forgets to trains, and goes directly to predict"""
    pass

class UnknownClassifierError(Exception):
    """raise this if user submits model out of model cabinet"""
    pass

logger = logging.getLogger(__name__)

class CompletionRiskModel:
    def __init__(self, *, model="logistic_regression", random_state=42):
        self.config = {
            "model_name": model,
            "random_state": random_state,
            "hyperparameters": {}
        }

        self.pipeline = None
        self.classifier = None
        self.is_trained = False

        self._initialize_classifier()

    def _initialize_classifier(self):
        name = self.config["model_name"]
        rand_stat = self.config["random_state"]

        model_cabinet = {
            "logistic_regression": LogisticRegression,
            "knn": KNeighborsClassifier,
            "random_forest": RandomForestClassifier
        }

        if name not in model_cabinet:
            raise UnknownClassifierError(f"Your requested model {name}, doesnt exist.")

        model_class = model_cabinet[name]

        if "random_state" in model_class.__init__.__code__.co_varnames:
            self.classifier = model_class(random_state=rand_stat)
        else:
            self.classifier = model_class()

        logger.info(f"Successfully initialized classifier, {name}")

        ## this shit works till now, please dont fuck it up!!

    def train(self, df):
        rand_stat = self.config["random_state"]
        
        target_df = defined_completion_risk(df)
        y = target_df["at_risk"]
        X = target_df.drop(columns=["at_risk"])

        pipeline = build_feature_pipeline()

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=rand_stat, stratify=y
        )

        X_train_trans = pipeline.fit_transform(X_train)
        X_test_trans = pipeline.transform(X_test)

        self.pipeline = pipeline # backup to class.

        self.classifier.fit(X_train_trans, y_train)

        train_acc = self.classifier.score(X_train_trans, y_train)
        test_acc = self.classifier.score(X_test_trans ,y_test)

        logging.info(
            f"Training Complete. Model:{self.config['model_name']}."
            f"Train accuracy -> {train_acc:.2f}, Test accuracy -> {test_acc:.2f}"
        )

        self.is_trained = True

    def predict(self, X_fresh):
        if not self.is_trained or self.pipeline is None:
            raise ModelNotTrainedError(f"Model has not been trained yet....")

        X_fresh_trans = self.pipeline.transform(X_fresh)

        predictions = self.classifier.predict(X_fresh_trans)

        return list(predictions)

    def save(self, filepath):
        if not self.is_trained:
            raise ModelNotTrainedError(f"Train and Predict your model first!!!!")

        artifact = {
            "config": self.config,
            "pipeline": self.pipeline,
            "classifier": self.classifier
        }

        joblib.dump(artifact, filepath)
        logger.info(f"Model + pipeline + config saved successfully to {filepath}")

    @classmethod
    def load(cls, filepath):
        artifact = joblib.load(filepath)
        saved_config = artifact["config"]

        instance = cls(
            model=saved_config["model_name"],
            random_state=saved_config["random_state"]
        )

        instance.config = saved_config
        instance.pipeline = artifact["pipeline"]
        instance.classifier = artifact["classifier"]
        instance.is_trained = True

        logger.info(f"Model artifact successfully restored from {filepath}. Ready to predict.")
        return instance

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, roc_curve

def evaluate_classifier(model, X_test, y_test) -> dict:
    
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_score = roc_auc_score(y_test, y_proba)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, label=f'ROC-AUC = {auc_score:.2f}')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend()
    plt.show()
    
    report = {
        "confusion_matrix": {"TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)},
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "roc_auc": auc_score
    }
    return report